# 02 — Unsupervised Clustering

KMeans and HDBSCAN over each cached embedding method (UMAP-reduced first),
scored against the hidden ground-truth labels via the Hungarian-matched
accuracy and standard clustering metrics.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import hdbscan
import pandas as pd
import umap
from sklearn.cluster import KMeans

from utils import config
from utils.data import stratified_sample
from utils.embeddings import load_cached
from utils.interpretability import summarize_clusters
from utils.metrics import evaluate_unsupervised

In [2]:
train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")

train_sample = stratified_sample(train_clean, config.SAMPLE_SIZE, seed=config.SEED)
suffix = f"n{config.SAMPLE_SIZE}" if config.SAMPLE_SIZE else "full"
true_labels = train_sample["label"].to_numpy()

# RoBERTa was embedded on a separate, smaller sample (see Task 7) — its
# true_labels must come from that same sample, not train_sample above.
roberta_train_sample = stratified_sample(train_clean, config.ROBERTA_SAMPLE_SIZE, seed=config.SEED)
roberta_suffix = f"n{config.ROBERTA_SAMPLE_SIZE}" if config.ROBERTA_SAMPLE_SIZE else "full"
roberta_true_labels = roberta_train_sample["label"].to_numpy()

METHODS = ["tfidf", "minilm", "roberta"]
if config.OPENAI_API_KEY and load_cached(f"openai_train_{suffix}") is not None:
    # Gate on cache existence, not just key presence — the OpenAI embedding
    # cell in 01_embeddings degrades gracefully on API errors (e.g. quota
    # exhaustion), so a configured key doesn't guarantee the cache exists.
    METHODS.append("openai")

embeddings_by_method = {}
labels_by_method = {}
texts_by_method = {}
for method in METHODS:
    method_suffix = roberta_suffix if method == "roberta" else suffix
    arr = load_cached(f"{method}_train_{method_suffix}")
    assert arr is not None, f"Missing cached embeddings for '{method}' — run 01_embeddings.ipynb first"
    embeddings_by_method[method] = arr
    labels_by_method[method] = roberta_true_labels if method == "roberta" else true_labels
    texts_by_method[method] = (roberta_train_sample if method == "roberta" else train_sample)["text"].tolist()
    assert arr.shape[0] == len(labels_by_method[method]), \
        f"{method}: embeddings ({arr.shape[0]} rows) and labels ({len(labels_by_method[method])} rows) size mismatch"

In [3]:
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = {}
runs = {}  # name -> (cluster_labels, emb_reduced, method) for the interpretability step below

for method, emb in embeddings_by_method.items():
    method_labels = labels_by_method[method]

    reducer = umap.UMAP(n_components=50, metric="cosine", random_state=config.SEED)
    emb_reduced = reducer.fit_transform(emb)

    kmeans = KMeans(n_clusters=config.NUM_CLASSES, random_state=config.SEED, n_init=10)
    km_labels = kmeans.fit_predict(emb_reduced)
    km_metrics = evaluate_unsupervised(method_labels, km_labels, emb_reduced)
    all_results[f"{method}_kmeans"] = km_metrics
    runs[f"{method}_kmeans"] = (km_labels, emb_reduced, method)

    clusterer = hdbscan.HDBSCAN(min_cluster_size=50, metric="euclidean",
                                 cluster_selection_method="eom")
    hdb_labels = clusterer.fit_predict(emb_reduced)
    hdb_metrics = evaluate_unsupervised(method_labels, hdb_labels, emb_reduced)
    all_results[f"{method}_hdbscan"] = hdb_metrics
    runs[f"{method}_hdbscan"] = (hdb_labels, emb_reduced, method)

    print(f"{method}: KMeans ACC={km_metrics['ACC (Hungarian)']:.3f} | "
          f"HDBSCAN coverage={hdb_metrics['Coverage']:.2f} ACC={hdb_metrics['ACC (Hungarian)']:.3f}")

for name, metrics in all_results.items():
    with open(config.RESULTS_DIR / f"metrics_{name}.json", "w") as f:
        json.dump(metrics, f, indent=2)

print(f"Saved {len(all_results)} result files to {config.RESULTS_DIR}")

C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


tfidf: KMeans ACC=0.808 | HDBSCAN coverage=0.96 ACC=0.262


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


minilm: KMeans ACC=0.830 | HDBSCAN coverage=1.00 ACC=0.493


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


roberta: KMeans ACC=0.518 | HDBSCAN coverage=0.00 ACC=0.000
Saved 6 result files to C:\Users\ACER\OneDrive\Documents\final-project\results


### Cluster inspection: what did each cluster actually find?

The metrics above (ACC, NMI, ...) are permutation-invariant number-matching
between cluster IDs and true label IDs — they don't show *what a cluster is
about*. A cluster whose top terms are e.g. "soccer, goal, league, match" is
obviously "Sports" to a human even if the numeric score alone doesn't make
that legible. This prints, per method+algorithm, each cluster's size,
majority true label, purity, top TF-IDF terms, and example documents nearest
its centroid — eval-only, not used anywhere as a training signal.

In [4]:
for name, (cluster_labels, emb_reduced, method) in runs.items():
    method_labels = labels_by_method[method]
    texts = texts_by_method[method]

    summary = summarize_clusters(
        texts, cluster_labels, method_labels, emb_reduced, config.CLASS_NAMES)

    print(f"\n=== {name} ===")
    if summary.empty:
        print("(no non-noise clusters — everything was noise)")
        continue
    with pd.option_context("display.max_colwidth", 60):
        print(summary.drop(columns="example_docs").to_string(index=False))
    for _, row in summary.iterrows():
        print(f"  cluster {row['cluster']} examples:")
        for doc in row["example_docs"]:
            print(f"    - {doc}")

    summary.to_csv(config.RESULTS_DIR / f"clusters_{name}.csv", index=False)

print(f"\nSaved per-cluster qualitative summaries to {config.RESULTS_DIR} (clusters_*.csv)")


=== tfidf_kmeans ===
 cluster  size majority_true_label   purity                                                                     top_terms
       0  2217              Sports 0.852954                        39, ap, game, season, quot, win, team, new, cup, night
       1  2020            Sci/Tech 0.760891 microsoft, 39, new, software, space, internet, company, google, music, online
       2  2169               World 0.775473                     iraq, gt, lt, 39, said, ap, president, reuters, bush, afp
       3  1594            Business 0.848181      oil, reuters, prices, stocks, 39, said, percent, fullquote, quarter, new
  cluster 0 examples:
    - Woburn's win is all Hart Reading fans knew Middlesex League rival Woburn had a sensational player in Boston College-boun
    - Pandolfo gives Rockets a lift When a tournament winds down to the final rounds, the teams with the best players usually 
    - Revolution hope history repeats itself Based on their finishes in the last two MLS reg


=== tfidf_hdbscan ===
 cluster  size majority_true_label   purity                                                         top_terms
       0   132            Sci/Tech 0.500000 lt, gt, font, strong, size, ms, 666666, helvetica, arial, verdana
       1  7529              Sports 0.257405     39, new, ap, reuters, said, quot, year, oil, world, microsoft
  cluster 0 examples:
    - STN-LCD maker EDT projects above 130 CSTN capacity growth in 2Q &lt;b&gt;...&lt;/b&gt; Emerging Display Technologies (ED
    - ATA Holdings Corp. Announces Definitive Agreement With AirTran &lt;b&gt;...&lt;/b&gt; INDIANAPOLIS, Nov. 16 /PRNewswire-
    - CIBC  #39;disturbed #39; at word of continued faxes of client data to US &lt;b&gt;...&lt;/b&gt; TORONTO (CP) - CIBC reve
  cluster 1 examples:
    - Nestle Considering Perrier Sale (AP) AP - Nestle SA said Wednesday it is considering selling French mineral water brand 
    - Governor Pledges to Ban Violent Video Games for Minors Governor Rod Blagojevich wants Ill


=== minilm_kmeans ===
 cluster  size majority_true_label   purity                                                              top_terms
       0  1993               World 0.869543  iraq, 39, said, ap, president, reuters, killed, afp, minister, people
       1  2032              Sports 0.958169               39, ap, game, season, win, team, cup, night, league, new
       2  1803            Sci/Tech 0.779257 microsoft, 39, new, software, internet, company, gt, lt, google, music
       3  2172            Business 0.716390           oil, reuters, 39, prices, new, said, stocks, lt, gt, percent
  cluster 0 examples:
    - Dhaka to announce reward for disclosing grenade attacker #39;s &lt;b&gt;...&lt;/b&gt; The Bangladeshi government may ann
    - Norway #39;s Riga embassy to reopen after alert Norway plans to reopen its embassy in Riga on Wednesday after a two-day 
    - Bangladesh government offers reward to catch grenade attackers (AFP) AFP - The Bangladeshi government has announced a 16


=== minilm_hdbscan ===
 cluster  size majority_true_label   purity                                                top_terms
       0  5968            Business 0.334115 39, reuters, said, new, gt, lt, ap, oil, microsoft, quot
       1  2032              Sports 0.958169 39, ap, game, season, win, team, cup, night, league, new
  cluster 0 examples:
    - China Bans a New Computer Video Game (AP) AP - China has banned a new computer game for referring to Taiwan and other re
    - McGuinty Wants Kids in Classroom  Not Malls Students in Ontario will have to keep their nose in their books until they #
    - WTO Eyes U.S. Offshore Web Gambling Ban (AP) AP - In a ruling that could open the United States to offshore Internet gam
  cluster 1 examples:
    - Hill out for nine months London - Richard Hill, a key member of the long-established England World Cup-winning back row,
    - Calgary teammates among CFL Players of the Week Toronto, ON (Sports Network) - Calgary Stampeders running back Joff